In [ ]:
import json
import os
from pathlib import Path

SOURCE_ROOT = Path('..')

with open('sentences.txt', 'a', encoding='utf-8') as fout:
    with open('sentences_lowercase.txt', 'a', encoding='utf-8') as fout_lowercase:
        for root, _, files in os.walk(SOURCE_ROOT):
            for file in files:
                if file.endswith('.align.tok.json'):
                    p = os.path.join(root, file)
                    with open(p, encoding='utf-8') as f:
                        data = json.load(f)
                    for d in data:
                        fout.write(d['text'] + '\n')
                        fout_lowercase.write(d['text'].lower() + '\n')

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input='sentences.txt',
    model_prefix='spm_unigram_16k',
    model_type='unigram',
    vocab_size=16384,
    character_coverage=1.0,
    shuffle_input_sentence=True,
    unk_id=0,
    unk_piece='<unk>',
    pad_id=1,
    pad_piece='<pad>',
    bos_id=2,
    bos_piece='<s>',
    eos_id=3,
    eos_piece='</s>',
)

spm.SentencePieceTrainer.Train(
    input='sentences_lowercase.txt',
    model_prefix='spm_unigram_lowercase_16k',
    model_type='unigram',
    vocab_size=16384,
    character_coverage=1.0,
    shuffle_input_sentence=True,
    unk_id=0,
    unk_piece='<unk>',
    pad_id=1,
    pad_piece='<pad>',
    bos_id=2,
    bos_piece='<s>',
    eos_id=3,
    eos_piece='</s>',
)


In [ ]:
sp = spm.SentencePieceProcessor()
sp.LoadFromFile('spm_unigram_16k.model')

sp_lowercase = spm.SentencePieceProcessor()
sp_lowercase.LoadFromFile('spm_unigram_lowercase_16k.model')

for root, _, files in os.walk(SOURCE_ROOT):
    for file in files:
        if file.endswith('.align.tok.json'):
            path = os.path.join(root, file)
            with open(path, encoding='utf-8') as fin:
                data = json.load(fin)
            for d in data:
                d['tokens'] = sp.Encode(d['text'])
                d['text_lower'] = d['text'].lower()
                d['tokens_lower'] = sp_lowercase.Encode(d['text_lower'])
            with open(path, 'w', encoding='utf-8') as fout:
                json.dump(data, fout, ensure_ascii=False, indent=2)